# Question 2: MLflow Experiment Comparison (MNIST MLP)

In this exercise, we adapt our previous training script to use an MLP on the MNIST dataset instead of Random Forest on Iris, and track at least 6 experiment runs in MLflow by varying hyperparameters like learning rate, batch size, and network architecture.

## Step 0 — Setup and Imports

In [1]:
import time
import warnings
warnings.filterwarnings('ignore')

import mlflow
import mlflow.sklearn
import pandas as pd
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, f1_score

mlflow.set_tracking_uri("http://localhost:5000")
mlflow.set_experiment("mnist-mlp-experiments")
print("Tracking URI:", mlflow.get_tracking_uri())

Tracking URI: http://localhost:5000


## Step 1 — Load MNIST Dataset

In [2]:
# Load MNIST dataset and normalize pixel values
X, y = fetch_openml('mnist_784', version=1, as_frame=False, return_X_y=True, parser='auto')
X = X / 255.0

# 30k training samples and 10k validation samples
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=10000, train_size=30000, random_state=42, stratify=y
)
print("Train shape:", X_train.shape, "Val shape:", X_val.shape)

Train shape: (30000, 784) Val shape: (10000, 784)


## Step 2 — Training & MLflow Logging Function

In [3]:
def train_and_log(hidden_layer_sizes=(128,), lr=0.001, batch_size=128, max_iter=15, run_name="mlp-run"):
    with mlflow.start_run(run_name=run_name):
        # Log parameters
        mlflow.log_param("hidden_layer_sizes", str(hidden_layer_sizes))
        mlflow.log_param("learning_rate_init", lr)
        mlflow.log_param("batch_size", batch_size)
        mlflow.log_param("max_iter", max_iter)
        mlflow.log_param("solver", "adam")
        mlflow.log_param("random_state", 42)

        # Train MLP
        model = MLPClassifier(
            hidden_layer_sizes=hidden_layer_sizes,
            learning_rate_init=lr,
            batch_size=batch_size,
            max_iter=max_iter,
            random_state=42
        )
        t0 = time.time()
        model.fit(X_train, y_train)
        train_time = time.time() - t0

        # Evaluate
        preds = model.predict(X_val)
        acc = accuracy_score(y_val, preds)
        f1 = f1_score(y_val, preds, average="macro")
        train_loss = float(model.loss_)

        # Log metrics and artifacts
        mlflow.log_metric("train_loss", train_loss)
        mlflow.log_metric("val_accuracy", acc)
        mlflow.log_metric("val_f1_macro", f1)
        mlflow.log_metric("train_time_sec", train_time)

        mlflow.set_tag("dataset", "MNIST")
        mlflow.sklearn.log_model(model, name="model", serialization_format="cloudpickle")

        run_id = mlflow.active_run().info.run_id
        print(f"{run_name} -> val_acc: {acc:.4f}, train_loss: {train_loss:.4f}, time: {train_time:.1f}s")
        return run_id

## Step 3 — Run 6 Experiments (Hyperparameter Sweep)

In [4]:
experiments = [
    {"name": "mlp-baseline-128", "hidden_layer_sizes": (128,), "lr": 0.001, "batch_size": 128},
    {"name": "mlp-deep-128-64", "hidden_layer_sizes": (128, 64), "lr": 0.001, "batch_size": 128},
    {"name": "mlp-wide-256-128", "hidden_layer_sizes": (256, 128), "lr": 0.001, "batch_size": 128},
    {"name": "mlp-high-lr-0.01", "hidden_layer_sizes": (128,), "lr": 0.01, "batch_size": 128},
    {"name": "mlp-low-lr-0.0001", "hidden_layer_sizes": (128,), "lr": 0.0001, "batch_size": 128},
    {"name": "mlp-small-batch-64", "hidden_layer_sizes": (128,), "lr": 0.001, "batch_size": 64},
]

run_ids = []
for exp in experiments:
    rid = train_and_log(
        hidden_layer_sizes=exp["hidden_layer_sizes"],
        lr=exp["lr"],
        batch_size=exp["batch_size"],
        max_iter=15,
        run_name=exp["name"]
    )
    run_ids.append(rid)

2026/08/23 05:27:35 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


mlp-baseline-128 -> val_acc: 0.9685, train_loss: 0.0229, time: 24.1s
🏃 View run mlp-baseline-128 at: http://localhost:5000/#/experiments/2/runs/8a07df72d3724ea8ab9a94e48ecd920d
🧪 View experiment at: http://localhost:5000/#/experiments/2


2026/08/23 05:28:11 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


mlp-deep-128-64 -> val_acc: 0.9705, train_loss: 0.0080, time: 29.7s
🏃 View run mlp-deep-128-64 at: http://localhost:5000/#/experiments/2/runs/6ebd07dffe6a4394a0225295add3f049
🧪 View experiment at: http://localhost:5000/#/experiments/2


2026/08/23 05:29:10 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


...

mlp-high-lr-0.01 -> val_acc: 0.9593, train_loss: 0.0344, time: 21.2s
🏃 View run mlp-high-lr-0.01 at: http://localhost:5000/#/experiments/2/runs/34e8a9de3d3240afbed745d4c2e2d0cc
🧪 View experiment at: http://localhost:5000/#/experiments/2


2026/08/23 05:30:05 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


mlp-low-lr-0.0001 -> val_acc: 0.9402, train_loss: 0.2025, time: 22.2s
🏃 View run mlp-low-lr-0.0001 at: http://localhost:5000/#/experiments/2/runs/3558ebeb3bc446f5bd41559150b788a7
🧪 View experiment at: http://localhost:5000/#/experiments/2


2026/08/23 05:30:51 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


mlp-small-batch-64 -> val_acc: 0.9689, train_loss: 0.0107, time: 40.3s
🏃 View run mlp-small-batch-64 at: http://localhost:5000/#/experiments/2/runs/355c842ec6664e188c363ea1813ebd9e
🧪 View experiment at: http://localhost:5000/#/experiments/2


## Step 4 — Compare Runs with `mlflow.search_runs()`

In [5]:
runs_df = mlflow.search_runs(
    experiment_names=["mnist-mlp-experiments"],
    order_by=["metrics.val_accuracy DESC"]
)

cols = [
    "tags.mlflow.runName",
    "params.hidden_layer_sizes",
    "params.learning_rate_init",
    "params.batch_size",
    "metrics.train_loss",
    "metrics.val_accuracy",
    "metrics.val_f1_macro",
    "metrics.train_time_sec",
    "run_id"
]

results = runs_df[[c for c in cols if c in runs_df.columns]].copy()
results.columns = [c.split('.')[-1] for c in results.columns]
print(results.to_string(index=False))

           runName hidden_layer_sizes learning_rate_init batch_size  train_loss  val_accuracy  val_f1_macro  train_time_sec                           run_id
  mlp-wide-256-128         (256, 128)              0.001        128    0.006909        0.9721      0.971891       52.903769 cfcaa12fc49648fe9921f4791b31a7ac
  mlp-wide-256-128         (256, 128)              0.001        128    0.006909        0.9721      0.971891       49.612566 b96a1bd934f64c34bdd49f1cfbae9209
   mlp-deep-128-64          (128, 64)              0.001        128    0.007975        0.9705      0.970275       29.724549 6ebd07dffe6a4394a0225295add3f049
mlp-small-batch-64             (128,)              0.001         64    0.010652        0.9689      0.968702       40.341558 355c842ec6664e188c363ea1813ebd9e
mlp-small-batch-64             (128,)              0.001         64    0.010652        0.9689      0.968702       46.433409 2438fe8ccbeb46fb912337db23a60ea0
  mlp-baseline-128             (128,)              0.001  